In [ ]:
!pip install -q -U google-generativeai langchain-google-genai langchain chromadb pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 3939_ESCI2026_CRC.pdf (1).pdf to 3939_ESCI2026_CRC.pdf (1).pdf


In [ ]:
import os
import sys
import google.generativeai as genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA

# 1. SETUP
API_KEY = "GEMINI API KEY"
os.environ["GOOGLE_API_KEY"] = API_KEY
genai.configure(api_key=API_KEY)

def run_demo():
    file_path = "/content/3939_ESCI2026_CRC.pdf (1).pdf"

    print("🔍 Checking 2026 API Permissions...")
    try:
        # Check what models are active for your key
        active_models = [m.name for m in genai.list_models()]
        print(f"✅ Active models found: {active_models[:5]}...") # Showing first 5
    except Exception as e:
        print(f"❌ API Connection Failed: {e}")
        return

    print("\n📄 Step 1: Loading PDF...")
    loader = PyPDFLoader(file_path)
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(loader.load())

    print("🧠 Step 2: Generating Embeddings...")
    # Use the stable embedding model for 2026
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    vectorstore = Chroma.from_documents(chunks, embeddings)
    print("   ✅ Embeddings Ready!")

    print("🤖 Step 3: Running Q&A...")
    # UPDATED: Use Gemini 2.5 Flash (the 2026 stable version)
    # If this fails, try "models/gemini-3-flash"
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
    )

    query = "What is the main objective of this research?"
    print(f"\n❓ Question: {query}")
    print("⏳ Processing (waiting for Gemini 2.5 response)...")
    sys.stdout.flush()

    try:
        response = qa.invoke(query)
        print("\n" + "="*40)
        print(f"💡 FINAL OUTPUT:\n\n{response['result']}")
        print("="*40)
    except Exception as e:
        print(f"❌ Error during generation: {e}")

run_demo()

🔍 Checking 2026 API Permissions...
✅ Active models found: ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-exp-image-generation']...

📄 Step 1: Loading PDF...
🧠 Step 2: Generating Embeddings...
   ✅ Embeddings Ready!
🤖 Step 3: Running Q&A...

❓ Question: What is the main objective of this research?
⏳ Processing (waiting for Gemini 2.5 response)...

💡 FINAL OUTPUT:

The main objective of this research is to bridge the research gap in exploring sarcasm-aware sentiment analysis for Hindi–Marathi social media text using Indic-specific transformer models. It aims to achieve this by evaluating a unified transformer-based approach that focuses on contextual understanding, multilingual representation, and balanced class detection.
